In [0]:
dbutils.widgets.text("api_key", "", "Enter your OpenAQ API key here")


In [0]:
api_key = dbutils.widgets.get("api_key")
print("API Key starts with:", api_key[:4] + "..." if api_key else "No key entered")

In [0]:
%pip install requests pandas
dbutils.library.restartPython()


In [0]:
import requests
import pandas as pd
import time


In [0]:
api_key = dbutils.widgets.get("api_key")
if not api_key:
    raise ValueError("⚠️ Please enter your OpenAQ API key in the widget above")


In [0]:
HEADERS = {
    "Accept": "application/json",
    "x-api-key": api_key
}


In [0]:
import requests, time

def fetch_india_locations(url, params, max_pages=20):
    results = []
    page = 1
    while page <= max_pages:
        params["page"] = page
        params["limit"] = 100
        res = requests.get(url, headers=HEADERS, params=params)

        if res.status_code == 429:
            print("Rate limit reached — waiting 30 seconds...")
            time.sleep(30)
            continue

        res.raise_for_status()
        data = res.json()
        batch = data.get("results", [])

        # Filter for India timezone
        india_batch = [loc for loc in batch if loc.get("timezone") == "Asia/Kolkata"]
        results.extend(india_batch)

        print(f"Fetched page {page}, batch size: {len(batch)}, India locations: {len(india_batch)}")

        if len(batch) < 100:
            break

        page += 1
        time.sleep(1)

    print(f"Total India locations found: {len(results)}")
    return results


In [0]:
BASE_URL = "https://api.openaq.org/v3"
url = f"{BASE_URL}/locations"
params = {"parameter": "pm25"}  # fetching PM2.5 now

india_locations = fetch_india_locations(url, params, max_pages=20)


In [0]:
#checking for all the found locations

import pandas as pd

# Convert the list of dictionaries to a DataFrame
india_locations_df = pd.DataFrame(india_locations)

# Display all columns for clarity
pd.set_option('display.max_columns', None)

# Show the DataFrame
display(india_locations_df)


In [0]:
#fileterd the data using delhi's longitude and lattitude coordinates

DELHI_BOUNDS = {
    "lat_min": 28.40,
    "lat_max": 28.88,
    "lon_min": 76.84,
    "lon_max": 77.35
}

delhi_locations = [
    loc for loc in india_locations
    if "coordinates" in loc
    and DELHI_BOUNDS["lat_min"] <= loc["coordinates"]["latitude"] <= DELHI_BOUNDS["lat_max"]
    and DELHI_BOUNDS["lon_min"] <= loc["coordinates"]["longitude"] <= DELHI_BOUNDS["lon_max"]
]

print(f"Delhi locations found: {len(delhi_locations)}")


In [0]:
import requests
import time
import pandas as pd

# 1. Extract Unique Location IDs (Assuming 'delhi_locations' is correctly populated)
delhi_location_ids = [loc.get('id') for loc in delhi_locations]
print(f"Unique Delhi Location IDs for query: {len(delhi_location_ids)}")

# --- Configuration (using existing variables) ---
# BASE_URL = "https://api.openaq.org/v3"
# HEADERS = { "Accept": "application/json", "x-api-key": api_key }


# --- STEP 1: Find PM2.5 Sensor IDs using /locations/{id} endpoint ---
pm25_sensor_ids = set()
total_locations = len(delhi_location_ids)

for i, loc_id in enumerate(delhi_location_ids):
    # Construct the correct URL path: /v3/locations/{location_id}
    LOCATION_DETAIL_URL = f"{BASE_URL}/locations/{loc_id}"
    
    print(f"Fetching sensor details for ID {loc_id} ({i+1}/{total_locations})...")
    
    try:
        res = requests.get(LOCATION_DETAIL_URL, headers=HEADERS)
        res.raise_for_status()
        data = res.json()
        
        if data.get("results"):
            location_details = data["results"][0]
            
            # 2. Iterate through sensors and filter for PM2.5 (Parameter ID 2)
            for sensor in location_details.get("sensors", []):
                param_id = sensor.get("parameter", {}).get("id")
                
                # PM2.5 has parameter ID 2 in OpenAQ
                if param_id == 2: 
                    # 3. Collect the sensor ID
                    pm25_sensor_ids.add(sensor.get('id'))
                    break # Found the PM2.5 sensor, move to the next location
            
    except requests.exceptions.RequestException as e:
        print(f"Error fetching location detail for ID {loc_id}: {e}")
        
    time.sleep(0.5) 

print(f"\nFound {len(pm25_sensor_ids)} unique PM2.5 sensor IDs.")
# Assuming pm25_sensor_ids is correctly populated from the previous successful step.

# --- Configuration (using existing variables) ---
# BASE_URL = "https://api.openaq.org/v3"
# HEADERS = { "Accept": "application/json", "x-api-key": api_key }


# Assuming pm25_sensor_ids is correctly populated from the successful Step 1.

# --- STEP 2: Query /latest endpoint and filter by Sensor ID (Correct Structure) ---
final_pm25_records = []
total_locations = len(delhi_location_ids)

for i, loc_id in enumerate(delhi_location_ids):
    # Construct the correct URL path: /v3/locations/{location_id}/latest
    LATEST_LOCATION_URL = f"{BASE_URL}/locations/{loc_id}/latest"
    
    print(f"Fetching ALL latest data for ID {loc_id} ({i+1}/{total_locations}) from {LATEST_LOCATION_URL}...")
    
    try:
        res = requests.get(LATEST_LOCATION_URL, headers=HEADERS)
        res.raise_for_status() 
        data = res.json()
        
        # 🐛 FIX: Iterate directly over the 'results' array, as each result IS a measurement
        for measurement in data.get("results", []):
            
            # 🐛 FIX: The Sensor ID is now accessed as a top-level key 'sensorsId' 
            # (as per the V3 documentation example)
            sensor_id = measurement.get("sensorsId") 
            
            # Filter to keep only latest info for sensor ids that exist in the pm25 sensor id list
            if sensor_id in pm25_sensor_ids:
                # This is a valid PM2.5 reading from a known Delhi sensor
                final_pm25_records.append({
                    "id": loc_id,
                    "pm25_value": measurement.get("value"),
                    "pm25_unit": "µg/m³", # Unit must be hardcoded or looked up since it's missing in /latest
                    "last_updated": measurement.get("datetime", {}).get("utc")
                })
                # Since the latest endpoint returns a single measurement per parameter, 
                # we can break after finding the first match for this location ID.
                break
        
    except requests.exceptions.RequestException as e:
        print(f"Error fetching latest data for ID {loc_id}: {e}")
        
    time.sleep(0.5) 

latest_pm25_df = pd.DataFrame(final_pm25_records)


# -------------------------------------------------------------
# CHECKPOINT: Display the data before the final merge
print("\n--- Fetched latest_pm25_df (Preview) ---")
display(latest_pm25_df) 
# -------------------------------------------------------------



In [0]:

# --- STEP 3: Final Merge with original Location Metadata ---

print("\n--- Final pm25_df ---")

# Create a base DataFrame from the successful location pull (delhi_locations) 
base_df = pd.DataFrame(delhi_locations)

# Correctly select and rename columns for the final DataFrame structure
base_df = base_df[['id', 'name', 'locality', 'coordinates']].drop_duplicates(subset=['id'])
base_df['latitude'] = base_df['coordinates'].apply(lambda x: x.get('latitude'))
base_df['longitude'] = base_df['coordinates'].apply(lambda x: x.get('longitude'))
base_df = base_df.rename(columns={'name': 'location', 'locality': 'city'})[['id', 'location', 'city', 'latitude', 'longitude']]


# Merge the base location info with the latest PM2.5 data
pm25_df = pd.merge(
    base_df,
    latest_pm25_df[['id', 'pm25_value', 'pm25_unit', 'last_updated']],
    on='id',
    how='left' 
)

display(pm25_df)

In [0]:
import pandas as pd

DANGER_LIMIT = 60  # WHO 24h guideline for PM2.5

pm25_df["danger_status"] = pm25_df["pm25_value"].apply(
    lambda x: "Cannot be determined" if pd.isnull(x)
    else ("Dangerous" if x > DANGER_LIMIT else "Safe")
)

display(pm25_df)


In [0]:
not_safe_df = pm25_df[pm25_df["danger_status"] == "Dangerous"]

display(not_safe_df)

In [0]:
safe_df = pm25_df[pm25_df["danger_status"] == "Safe"]
display(safe_df)

In [0]:
undetermined_df = pm25_df[pm25_df["danger_status"] == "Cannot be determined"]
display(undetermined_df)